# Mini-Projeto 1 — SBC de Regras IF-THEN
## Domínio: Triagem Médica Simples

**Sistema Baseado em Conhecimento** que simula a triagem médica de um paciente utilizando regras IF-THEN com encadeamento progressivo e resolução de conflitos por salience (prioridade).

---
### Estrutura do encadeamento (3 níveis):
- **Nível 1 (Fatos):** Sintomas brutos do paciente (febre, tosse, dor, etc.)
- **Nível 2 (Hipóteses):** Condições inferidas (suspeita_gripe, suspeita_covid, etc.)
- **Nível 3 (Decisão):** Recomendação final (repouso, UPA, hospital urgente, etc.)

## 1. Motor de Inferência (Mecanismo do SBC)

In [ ]:
class SistemaEspecialista:
    """
    Motor de inferência por encadeamento progressivo (forward chaining).
    Cada regra possui um atributo 'salience' (prioridade) para resolução de conflitos.
    Regras com maior salience são avaliadas primeiro quando múltiplas podem disparar.
    """

    def __init__(self):
        self.fatos = {}        # Base de fatos (memória de trabalho)
        self.trace = []        # Rastro de decisões para explicação
        self.regras_disparadas = []

    def definir_fatos(self, fatos_iniciais: dict):
        """Carrega os fatos iniciais (sintomas do paciente)."""
        self.fatos = fatos_iniciais.copy()
        self.trace = []
        self.regras_disparadas = []
        self.trace.append("=" * 60)
        self.trace.append("INÍCIO DA SESSÃO DE TRIAGEM")
        self.trace.append(f"Fatos iniciais: {self.fatos}")
        self.trace.append("=" * 60)

    def checar_condicao(self, condicoes: dict) -> bool:
        """Verifica se todas as condições de uma regra são satisfeitas pelos fatos atuais."""
        for chave, valor in condicoes.items():
            if chave.startswith("NOT_"):
                # Resolução de conflito com NOT: a chave real não deve estar nos fatos como True
                chave_real = chave[4:]
                if self.fatos.get(chave_real) == True:
                    return False
            else:
                if self.fatos.get(chave) != valor:
                    return False
        return True

    def disparar_regra(self, nome: str, conclusao: dict, explicacao: str):
        """Adiciona a conclusão de uma regra à base de fatos e registra no trace."""
        if nome not in self.regras_disparadas:
            self.fatos.update(conclusao)
            self.regras_disparadas.append(nome)
            self.trace.append(f"\n>>> REGRA DISPARADA: {nome}")
            self.trace.append(f"    Conclusão: {conclusao}")
            self.trace.append(f"    Explicação: {explicacao}")

    def inferir(self):
        """Executa o motor de inferência aplicando todas as regras em ordem de prioridade."""

        # ============================================================
        # REGRAS definidas como lista ordenada por SALIENCE (prioridade)
        # Maior salience = avaliada primeiro (resolução de conflito)
        # ============================================================
        regras = [

            # ----------------------------------------------------------
            # NÍVEL 1 → NÍVEL 2: Sintomas → Hipóteses
            # ----------------------------------------------------------
            {
                "nome": "R1 - Suspeita de Gripe",
                "salience": 10,
                "condicoes": {"febre": True, "tosse": True, "dor_corpo": True},
                "conclusao": {"suspeita_gripe": True},
                "explicacao": "Febre + Tosse + Dor no corpo → suspeita de gripe"
            },
            {
                "nome": "R2 - Suspeita de COVID-19",
                "salience": 10,
                "condicoes": {"febre": True, "perda_olfato": True},
                "conclusao": {"suspeita_covid": True},
                "explicacao": "Febre + Perda de olfato → suspeita de COVID-19"
            },
            {
                "nome": "R3 - Suspeita de Infecção Bacteriana",
                "salience": 10,
                "condicoes": {"febre_alta": True, "dor_garganta": True, "NOT_tosse": True},
                "conclusao": {"suspeita_bacteriana": True},
                "explicacao": "Febre alta + Dor de garganta SEM tosse → suspeita bacteriana (NOT usado)"
            },
            {
                "nome": "R4 - Risco Cardiovascular",
                "salience": 10,
                "condicoes": {"dor_peito": True, "falta_ar": True},
                "conclusao": {"risco_cardiovascular": True},
                "explicacao": "Dor no peito + Falta de ar → possível emergência cardiovascular"
            },
            {
                "nome": "R5 - Quadro Leve",
                "salience": 5,
                "condicoes": {"febre": True, "NOT_falta_ar": True, "NOT_dor_peito": True},
                "conclusao": {"quadro_leve": True},
                "explicacao": "Febre SEM falta de ar e SEM dor no peito → quadro leve (NOT usado)"
            },

            # ----------------------------------------------------------
            # NÍVEL 2 → NÍVEL 3: Hipóteses → Decisão / Recomendação
            # ----------------------------------------------------------
            {
                "nome": "R6 - Repouso em Casa (Gripe Leve)",
                "salience": 8,
                "condicoes": {"suspeita_gripe": True, "quadro_leve": True},
                "conclusao": {"recomendacao": "Repouso em casa, hidratação e analgésico"},
                "explicacao": "Suspeita de gripe em quadro leve → tratamento domiciliar"
            },
            {
                "nome": "R7 - Teste COVID e Isolamento",
                "salience": 9,
                "condicoes": {"suspeita_covid": True, "NOT_falta_ar": True},
                "conclusao": {"recomendacao": "Realizar teste COVID, isolamento imediato"},
                "explicacao": "Suspeita de COVID sem falta de ar → teste + isolamento (NOT usado)"
            },
            {
                "nome": "R8 - Encaminhamento à UPA (Bacteriana)",
                "salience": 9,
                "condicoes": {"suspeita_bacteriana": True},
                "conclusao": {"recomendacao": "Ir à UPA para avaliação e possível antibiótico"},
                "explicacao": "Suspeita bacteriana → necessita avaliação médica presencial"
            },
            {
                "nome": "R9 - Emergência Cardiovascular",
                "salience": 100,  # PRIORIDADE MÁXIMA
                "condicoes": {"risco_cardiovascular": True},
                "conclusao": {"recomendacao": "CHAMAR SAMU (192) IMEDIATAMENTE"},
                "explicacao": "Risco cardiovascular identificado → emergência máxima (salience 100)"
            },
            {
                "nome": "R10 - COVID Grave com Falta de Ar",
                "salience": 50,  # PRIORIDADE ALTA
                "condicoes": {"suspeita_covid": True, "falta_ar": True},
                "conclusao": {"recomendacao": "Ir ao HOSPITAL URGENTE - possível COVID grave"},
                "explicacao": "COVID suspeito com falta de ar → hospitalização urgente (salience 50)"
            },
        ]

        # Ordena regras por salience decrescente (resolução de conflito)
        regras_ordenadas = sorted(regras, key=lambda r: r["salience"], reverse=True)

        self.trace.append("\n--- INICIANDO ENCADEAMENTO PROGRESSIVO ---")

        # Loop de inferência: continua enquanto alguma regra nova disparar
        houve_mudanca = True
        iteracao = 0
        while houve_mudanca:
            houve_mudanca = False
            iteracao += 1
            self.trace.append(f"\n[Iteração {iteracao}]")
            for regra in regras_ordenadas:
                if regra["nome"] not in self.regras_disparadas:
                    if self.checar_condicao(regra["condicoes"]):
                        self.disparar_regra(
                            regra["nome"],
                            regra["conclusao"],
                            regra["explicacao"]
                        )
                        houve_mudanca = True

        self.trace.append("\n" + "=" * 60)
        self.trace.append("FIM DO ENCADEAMENTO")
        self.trace.append(f"Total de regras disparadas: {len(self.regras_disparadas)}")
        self.trace.append("=" * 60)

    def exibir_trace(self):
        """Imprime o rastro completo de decisões."""
        for linha in self.trace:
            print(linha)

    def resultado(self):
        """Retorna a recomendação final."""
        return self.fatos.get("recomendacao", "Nenhuma recomendação gerada. Consulte um médico.")


print("✅ Motor de inferência carregado com sucesso!")

## 2. Casos de Teste

Abaixo são apresentados os **3 casos de teste** obrigatórios, cada um ativando um caminho diferente de encadeamento.

### Caso 1 — Paciente com Gripe Leve
**Sintomas:** febre, tosse, dor no corpo (SEM falta de ar, SEM dor no peito)

**Encadeamento esperado:**
1. R1 → `suspeita_gripe = True`
2. R5 → `quadro_leve = True` (via NOT)
3. R6 → Recomendação: repouso em casa

In [ ]:
sbc = SistemaEspecialista()

fatos_caso1 = {
    "febre": True,
    "tosse": True,
    "dor_corpo": True,
    "falta_ar": False,
    "dor_peito": False,
    "perda_olfato": False,
    "febre_alta": False,
    "dor_garganta": False
}

sbc.definir_fatos(fatos_caso1)
sbc.inferir()
sbc.exibir_trace()

print(f"\n🩺 RECOMENDAÇÃO FINAL: {sbc.resultado()}")

### Caso 2 — Suspeita de COVID-19 Grave
**Sintomas:** febre, perda de olfato, falta de ar

**Encadeamento esperado:**
1. R2 → `suspeita_covid = True`
2. R10 (salience 50) → Hospitalização urgente (tem prioridade sobre R7)

In [ ]:
sbc2 = SistemaEspecialista()

fatos_caso2 = {
    "febre": True,
    "perda_olfato": True,
    "falta_ar": True,
    "tosse": False,
    "dor_corpo": False,
    "dor_peito": False,
    "febre_alta": False,
    "dor_garganta": False
}

sbc2.definir_fatos(fatos_caso2)
sbc2.inferir()
sbc2.exibir_trace()

print(f"\n🩺 RECOMENDAÇÃO FINAL: {sbc2.resultado()}")

### Caso 3 — Emergência Cardiovascular
**Sintomas:** dor no peito, falta de ar

**Encadeamento esperado:**
1. R4 → `risco_cardiovascular = True`
2. R9 (salience 100, MÁXIMA PRIORIDADE) → Chamar SAMU imediatamente

In [ ]:
sbc3 = SistemaEspecialista()

fatos_caso3 = {
    "febre": False,
    "tosse": False,
    "dor_corpo": False,
    "dor_peito": True,
    "falta_ar": True,
    "perda_olfato": False,
    "febre_alta": False,
    "dor_garganta": False
}

sbc3.definir_fatos(fatos_caso3)
sbc3.inferir()
sbc3.exibir_trace()

print(f"\n🩺 RECOMENDAÇÃO FINAL: {sbc3.resultado()}")

## 3. Modo Interativo — Insira seus próprios sintomas

In [ ]:
def perguntar_sim_nao(pergunta):
    resp = input(f"{pergunta} (s/n): ").strip().lower()
    return resp == 's'

print("=" * 50)
print("   TRIAGEM MÉDICA INTERATIVA - SBC IF-THEN")
print("=" * 50)
print("Responda s (sim) ou n (não) para cada sintoma:\n")

fatos_usuario = {
    "febre":        perguntar_sim_nao("Tem febre (até 39°C)?"),
    "febre_alta":   perguntar_sim_nao("Tem febre alta (acima de 39°C)?"),
    "tosse":        perguntar_sim_nao("Tem tosse?"),
    "dor_corpo":    perguntar_sim_nao("Tem dor no corpo?"),
    "perda_olfato": perguntar_sim_nao("Perdeu o olfato ou paladar?"),
    "dor_garganta": perguntar_sim_nao("Tem dor de garganta?"),
    "dor_peito":    perguntar_sim_nao("Tem dor no peito?"),
    "falta_ar":     perguntar_sim_nao("Tem falta de ar?"),
}

sbc_interativo = SistemaEspecialista()
sbc_interativo.definir_fatos(fatos_usuario)
sbc_interativo.inferir()
sbc_interativo.exibir_trace()

print(f"\n🩺 RECOMENDAÇÃO FINAL: {sbc_interativo.resultado()}")
print("\n⚠️ Este sistema é apenas educacional. Consulte sempre um médico.")